# ARC + MolFormer — ChEMBL & BindingDB Pipeline (v10 — ALL BUGS FIXED)
## Adaptive Retention & Correction for Molecular Target Classification

### Based on:
- **Paper**: "Adaptive Retention & Correction: Test-Time Training for Continual Learning" (ICLR 2025)
- **Backbone**: IBM MolFormer-XL (`ibm/MoLFormer-XL-both-10pct`)
- **Datasets**: ChEMBL 15-class (`chembl_15class_final.csv`) + BindingDB 15-class (`bindingdb_15class.csv`)

---

### Bugs Fixed vs v9
| # | Bug | Where | Impact | Fix |
|---|-----|--------|--------|-----|
| **FIX-1** | `deepcopy` inside `evaluate_task` discarded all retention updates | Step 12 `evaluate_task` | Retention was completely no-op | Removed deepcopy; persistent_clf_arc shared across all eval calls |
| **FIX-2** | `c_hat` used softmax over past-logits slice only | `out_of_task_detection` Assumption 2 | w=c/c_hat inflated → correction fired on wrong samples → AUROC collapse | c_hat now computed from full softmax, max over past indices |
| **FIX-3** | Diagnostic cell computed w with same wrong c_hat formula | Diagnostic block in Step 12 | Diagnostic was mis-reporting w distribution | Fixed to match corrected OTD formula |
| **FIX-4** | `persistent_clf_arc` snapshot taken AFTER diagnostic but BEFORE eval loop | `run_cil_pipeline` | Snapshot was correct but deepcopy in eval killed it | Now only one path: no deepcopy in evaluate_task, snapshot set once per task |
| **FIX-5** | Step 11 `evaluate_task` defined then immediately redefined in Step 12 | Two function definitions | Python uses Step 12 version; Step 11 calls `compute_batch_theta` which doesn't exist | Removed dead Step 11 definition entirely |


## Step 1 — Install & Imports

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'pandas>=2.0.0',
    'rdkit',
    'einops',
    'rotary-embedding-torch',
    'scipy',
    'tqdm',
]
for pkg in packages:
    try:
        install(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg}: {e}')
print('\nInstallation complete.')

  ✓ torch>=2.0.0


  ✓ transformers>=4.35.0


  ✓ scikit-learn>=1.3.0


  ✓ numpy>=1.24.0


  ✓ matplotlib>=3.7.0


...

  ✓ rdkit


  ✓ einops


  ✓ rotary-embedding-torch


  ✓ scipy


  ✓ tqdm

Installation complete.


In [1]:
import os, json, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports OK')

Device: cuda
All imports OK


## Step 2 — Configuration

In [3]:
CFG = {
    # ── Data paths ────────────────────────────────────────────────────────
    'chembl_path'     : 'chembl_15class_final.csv',
    'bindingdb_path'  : 'bindingdb_15class.csv',
    'output_dir'      : 'arc_output_v10',

    # ── SMILES cleaning ───────────────────────────────────────────────────
    'min_ha'          : 5,
    'max_ha'          : 100,
    'min_mw'          : 100,
    'max_mw'          : 1500,

    # ── Sampling ──────────────────────────────────────────────────────────
    'max_samples'     : 30000,
    'seed'            : 42,

    # ── Train/val/test split ──────────────────────────────────────────────
    'train_ratio'     : 0.70,
    'val_ratio'       : 0.15,
    'test_ratio'      : 0.15,

    # ── CIL tasks ─────────────────────────────────────────────────────────
    'classes_per_task': 3,

    # ── MolFormer ─────────────────────────────────────────────────────────
    'molformer_name'  : 'ibm/MoLFormer-XL-both-10pct',
    'molformer_batch' : 32,
    'max_smiles_len'  : 202,

    # ── Classifier training ───────────────────────────────────────────────
    'clf_epochs'      : 100,
    'clf_lr'          : 1e-3,
    'clf_batch'       : 64,
    'es_patience'     : 10,
    'es_min_delta'    : 1e-4,

    # ── ARC hyperparameters ───────────────────────────────────────────────
    'arc_epsilon'     : 0.70,
    'arc_theta'       : 0.80,
    'arc_temp'        : 2.0,
    'arc_lr'          : 5e-4,
}

os.makedirs(CFG['output_dir'], exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

print('Config ready')
print(f"  classes_per_task = {CFG['classes_per_task']}  →  {15 // CFG['classes_per_task']} tasks on 15 classes")
print(f"  arc_epsilon={CFG['arc_epsilon']}  arc_theta={CFG['arc_theta']}  arc_temp={CFG['arc_temp']}")

Config ready
  classes_per_task = 3  →  5 tasks on 15 classes
  arc_epsilon=0.7  arc_theta=0.8  arc_temp=2.0


## Step 3 — Data Loading & SMILES Cleaning

In [4]:
def clean_smiles(smiles, cfg):
    if not isinstance(smiles, str) or smiles.strip() == '':
        return None, 'missing_or_empty'
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, 'invalid_smiles'
    try:
        mol = MolStandardize.rdMolStandardize.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    try:
        mol = MolStandardize.rdMolStandardize.Uncharger().uncharge(mol)
        mol = MolStandardize.rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    except Exception:
        pass
    try:
        ha = mol.GetNumHeavyAtoms()
        mw = rdMolDescriptors.CalcExactMolWt(mol)
        if not (cfg['min_ha'] <= ha <= cfg['max_ha'] and cfg['min_mw'] <= mw <= cfg['max_mw']):
            return None, 'filtered_out'
    except Exception:
        return None, 'filter_error'
    canon = Chem.MolToSmiles(mol, canonical=True)
    return (canon, 'ok') if canon else (None, 'canon_failed')


def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        sc = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(sc, canonical=True)
    except Exception:
        return smiles


print('SMILES cleaning helpers defined')

SMILES cleaning helpers defined


In [5]:
def load_and_clean_dataset(path, cfg, dataset_name='Dataset'):
    print(f'\n=== Loading {dataset_name} from {path} ===')
    df = pd.read_csv(path)
    print(f'  Raw shape: {df.shape}')
    print(f'  Labels: {sorted(df["label"].unique())}')

    if cfg.get('max_samples') is not None:
        n_classes = df['label'].nunique()
        per_class = cfg['max_samples'] // n_classes
        rng = np.random.default_rng(cfg['seed'])
        frames = []
        for cls in sorted(df['label'].unique()):
            cls_df = df[df['label'] == cls]
            n_take = min(per_class, len(cls_df))
            idx    = rng.choice(len(cls_df), size=n_take, replace=False)
            frames.append(cls_df.iloc[idx])
        df = pd.concat(frames).reset_index(drop=True)
        print(f'  After stratified subsample ({cfg["max_samples"]} total): {df.shape}')

    res = df['smiles'].apply(lambda s: clean_smiles(s, cfg))
    df['canon_smiles'] = res.apply(lambda x: x[0])
    df['clean_status'] = res.apply(lambda x: x[1])
    print(f'\n  Cleaning report ({dataset_name}):')
    print(df['clean_status'].value_counts().to_string())

    df = df[df['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
    df = df.rename(columns={'label': 'label_id'})
    print(f'  After cleaning: {df.shape}')

    df['scaffold'] = df['canon_smiles'].apply(get_scaffold)

    print(f'  Class distribution:')
    print(df['label_id'].value_counts().sort_index().to_string())
    return df


chembl_df    = load_and_clean_dataset(CFG['chembl_path'],    CFG, 'ChEMBL-15')
bindingdb_df = load_and_clean_dataset(CFG['bindingdb_path'], CFG, 'BindingDB-15')


=== Loading ChEMBL-15 from chembl_15class_final.csv ===
  Raw shape: (1103220, 2)
  Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  After stratified subsample (30000 total): (30000, 2)

  Cleaning report (ChEMBL-15):
clean_status
ok              29699
filtered_out      301
  After cleaning: (29681, 4)
  Class distribution:
label_id
0     1960
1     1981
2     1979
3     1971
4     1983
5     1980
6     1988
7     1975
8     1980
9     1975
10    1982
11    1976
12    1988
13    1986
14    1977

=== Loading BindingDB-15 from bindingdb_15class.csv ===
  Raw shape: (914051, 2)
  Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  After stratified subsample (30000 total): (30000, 2)

  Cleaning report (BindingDB-15):
clean_status
ok                29492
invalid_smiles      313
filtered_out        195
  After cleaning: (29466, 4)
  Class distribution:
label_id
0     1947
1     1964
2     1964
3     1964
4     1963
5     1969
6     1962
7     1961
8     1960
9     1968

## Step 4 — Label Maps & Stratified Scaffold Split

In [11]:
def stratified_scaffold_split(df, label_col, train_r, val_r, test_r, seed=42):
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for cls in sorted(df[label_col].unique()):
        cls_df = df[df[label_col] == cls]
        sc2idx = defaultdict(list)
        for idx, sc in zip(cls_df.index, cls_df['scaffold']):
            sc2idx[sc].append(idx)
        groups = list(sc2idx.values())
        rng.shuffle(groups)
        n_tr = max(1, int(len(cls_df) * train_r))
        n_vl = max(1, int(len(cls_df) * val_r))
        cls_tr, cls_vl, cls_ts = [], [], []
        for g in groups:
            if   len(cls_tr) < n_tr: cls_tr.extend(g)
            elif len(cls_vl) < n_vl: cls_vl.extend(g)
            else:                    cls_ts.extend(g)
        if len(cls_vl) == 0 and len(cls_tr) > 1: cls_vl.append(cls_tr.pop())
        if len(cls_ts) == 0 and len(cls_tr) > 1: cls_ts.append(cls_tr.pop())
        train_idx.extend(cls_tr); val_idx.extend(cls_vl); test_idx.extend(cls_ts)

    tr, vl, ts = df.loc[train_idx].copy(), df.loc[val_idx].copy(), df.loc[test_idx].copy()
    for sname, sdf in [('val', vl), ('test', ts)]:
        missing = set(df[label_col].unique()) - set(sdf[label_col].unique())
        if missing:
            print(f'  WARNING: {sname} missing classes {missing}')
    return tr, vl, ts


def verify_split(train_df, val_df, test_df, label_col='label_id', name=''):
    print(f'  {name} splits:')
    for sname, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        counts = sdf[label_col].value_counts().sort_index()
        print(f'    {sname}: {len(sdf)} samples | min_cls={counts.min()} max_cls={counts.max()}')
    tsc = set(train_df['scaffold']); vsc = set(val_df['scaffold']); esc = set(test_df['scaffold'])
    print(f'    Scaffold leakage → Train∩Val={len(tsc&vsc)} Train∩Test={len(tsc&esc)} Val∩Test={len(vsc&esc)}')


def build_label_map(df, label_col='label_id'):
    return {int(c): f'Target-{c}' for c in sorted(df[label_col].unique())}


print('\n=== ChEMBL Scaffold Split ===')
chembl_train, chembl_val, chembl_test = stratified_scaffold_split(
    chembl_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
verify_split(chembl_train, chembl_val, chembl_test, name='ChEMBL')
chembl_label_map = build_label_map(chembl_df)

print('\n=== BindingDB Scaffold Split ===')
bindingdb_train, bindingdb_val, bindingdb_test = stratified_scaffold_split(
    bindingdb_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
verify_split(bindingdb_train, bindingdb_val, bindingdb_test, name='BindingDB')
bindingdb_label_map = build_label_map(bindingdb_df)


=== ChEMBL Scaffold Split ===
  ChEMBL splits:
    Train: 20777 samples | min_cls=1372 max_cls=1394
    Val: 4449 samples | min_cls=294 max_cls=299
    Test: 4455 samples | min_cls=294 max_cls=299
    Scaffold leakage → Train∩Val=646 Train∩Test=616 Val∩Test=193

=== BindingDB Scaffold Split ===
  BindingDB splits:
    Train: 20622 samples | min_cls=1363 max_cls=1381
    Val: 4421 samples | min_cls=292 max_cls=299
    Test: 4423 samples | min_cls=292 max_cls=297
    Scaffold leakage → Train∩Val=712 Train∩Test=709 Val∩Test=215


## Step 5 — CIL Task Construction

In [12]:
def make_cil_tasks(train_df, val_df, test_df, cpt, label_map):
    all_ids = sorted(train_df['label_id'].unique())
    tasks = []
    for t in range(int(np.ceil(len(all_ids) / cpt))):
        new_cls  = all_ids[t*cpt : (t+1)*cpt]
        seen_cls = all_ids[: (t+1)*cpt]
        tasks.append({
            'task_id'    : t,
            'new_classes': new_cls,
            'all_classes': list(seen_cls),
            'n_classes'  : len(seen_cls),
            'class_names': {lid: label_map[lid] for lid in seen_cls},
            'train': train_df[train_df['label_id'].isin(seen_cls)].copy(),
            'val'  : val_df[val_df['label_id'].isin(seen_cls)].copy(),
            'test' : test_df[test_df['label_id'].isin(seen_cls)].copy(),
        })
    return tasks


chembl_tasks    = make_cil_tasks(chembl_train,    chembl_val,    chembl_test,    CFG['classes_per_task'], chembl_label_map)
bindingdb_tasks = make_cil_tasks(bindingdb_train, bindingdb_val, bindingdb_test, CFG['classes_per_task'], bindingdb_label_map)

print(f'ChEMBL CIL tasks: {len(chembl_tasks)}')
for t in chembl_tasks:
    print(f"  Task {t['task_id']}: new={[t['class_names'][c] for c in t['new_classes']]}  "
          f"train={len(t['train'])} val={len(t['val'])} test={len(t['test'])}")

ChEMBL CIL tasks: 5
  Task 0: new=['Target-0', 'Target-1', 'Target-2']  train=4143 val=887 test=890
  Task 1: new=['Target-3', 'Target-4', 'Target-5']  train=8296 val=1776 test=1782
  Task 2: new=['Target-6', 'Target-7', 'Target-8']  train=12455 val=2669 test=2673
  Task 3: new=['Target-9', 'Target-10', 'Target-11']  train=16609 val=3558 test=3563
  Task 4: new=['Target-12', 'Target-13', 'Target-14']  train=20777 val=4449 test=4455


## Step 6 — MolFormer Feature Extraction

In [13]:
print('Loading MolFormer tokenizer & model...')
tokenizer = AutoTokenizer.from_pretrained(CFG['molformer_name'], trust_remote_code=True)
molformer = AutoModel.from_pretrained(
    CFG['molformer_name'], trust_remote_code=True, deterministic_eval=True
)
molformer.eval().to(DEVICE)
FEAT_DIM = molformer.config.hidden_size

for p in molformer.parameters():
    p.requires_grad = False

print(f'MolFormer loaded on {DEVICE}  |  hidden_dim={FEAT_DIM}')
print('Backbone fully frozen (ARC paper Sec 3.2)')

Loading MolFormer tokenizer & model...
MolFormer loaded on cuda  |  hidden_dim=768
Backbone fully frozen (ARC paper Sec 3.2)


In [14]:
@torch.no_grad()
def extract_molformer_features(smiles_list, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc='MolFormer encode'):
        batch = smiles_list[i: i + batch_size]
        enc   = tokenizer(
            batch, padding=True, truncation=True,
            max_length=CFG['max_smiles_len'], return_tensors='pt'
        ).to(DEVICE)
        out    = molformer(**enc)
        hidden = out.last_hidden_state
        mask   = enc['attention_mask']
        mask_f = mask.unsqueeze(-1).float()
        summed = (hidden * mask_f).sum(dim=1)
        counts = mask_f.sum(dim=1).clamp(min=1)
        pooled = (summed / counts).cpu().numpy()
        all_emb.append(pooled)
    return np.vstack(all_emb)


print('Feature extractor ready')

Feature extractor ready


In [15]:
def extract_dataset_features(df, train_df, val_df, test_df, dataset_name='Dataset'):
    print(f'\n=== Extracting {dataset_name} features ({len(df)} molecules) ===')
    feats_all = extract_molformer_features(df['canon_smiles'].tolist(), CFG['molformer_batch'])
    idx_to_feat = {idx: feat for idx, feat in zip(df.index, feats_all)}

    def get_feats(split_df):
        X = np.stack([idx_to_feat[i] for i in split_df.index])
        y = split_df['label_id'].values
        return X, y

    X_tr, y_tr = get_feats(train_df)
    X_vl, y_vl = get_feats(val_df)
    X_te, y_te = get_feats(test_df)
    print(f'  train:{X_tr.shape} | val:{X_vl.shape} | test:{X_te.shape}')
    return X_tr, y_tr, X_vl, y_vl, X_te, y_te


chembl_X_tr, chembl_y_tr, chembl_X_vl, chembl_y_vl, chembl_X_te, chembl_y_te = \
    extract_dataset_features(chembl_df, chembl_train, chembl_val, chembl_test, 'ChEMBL')

bindingdb_X_tr, bindingdb_y_tr, bindingdb_X_vl, bindingdb_y_vl, bindingdb_X_te, bindingdb_y_te = \
    extract_dataset_features(bindingdb_df, bindingdb_train, bindingdb_val, bindingdb_test, 'BindingDB')


=== Extracting ChEMBL features (29681 molecules) ===


MolFormer encode:   0%|          | 0/928 [00:00<?, ?it/s]

  train:(20777, 768) | val:(4449, 768) | test:(4455, 768)

=== Extracting BindingDB features (29466 molecules) ===


MolFormer encode:   0%|          | 0/921 [00:00<?, ?it/s]

  train:(20622, 768) | val:(4421, 768) | test:(4423, 768)


## Step 7 — Expanding Linear Classifier & Replay Buffer

In [16]:
class MolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ExpandingLinearHead(nn.Module):
    def __init__(self, in_dim, n_init):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_init)

    def grow(self, n_new, device):
        old    = self.fc
        n_old  = old.out_features
        new_fc = nn.Linear(old.in_features, n_old + n_new)
        with torch.no_grad():
            new_fc.weight[:n_old] = old.weight
            new_fc.bias[:n_old]   = old.bias
            nn.init.xavier_uniform_(new_fc.weight[n_old:])
            nn.init.zeros_(new_fc.bias[n_old:])
        self.fc = new_fc.to(device)

    def forward(self, x): return self.fc(x)

    @property
    def n_classes(self): return self.fc.out_features


class ReplayBuffer:
    def __init__(self, max_per_class):
        self.max_per_class = max_per_class
        self._X = {}; self._y = {}; self._counts = {}

    def update(self, X, y, rng=None):
        if rng is None: rng = np.random.default_rng()
        for cls in np.unique(y):
            mask = y == cls
            if cls not in self._X:
                self._X[cls] = []; self._y[cls] = []; self._counts[cls] = 0
            for feat in X[mask]:
                n = self._counts[cls]
                if n < self.max_per_class:
                    self._X[cls].append(feat); self._y[cls].append(int(cls))
                else:
                    j = int(rng.integers(0, n + 1))
                    if j < self.max_per_class:
                        self._X[cls][j] = feat
                self._counts[cls] += 1

    def sample(self):
        if not self._X: return None, None
        X_p = np.vstack([np.stack(self._X[c]) for c in sorted(self._X)])
        y_p = np.concatenate([np.array(self._y[c]) for c in sorted(self._y)])
        return X_p, y_p

    def __len__(self): return sum(len(v) for v in self._X.values())


print('ExpandingLinearHead and ReplayBuffer defined')

ExpandingLinearHead and ReplayBuffer defined


In [17]:
def train_classifier_on_task(clf, X_tr, y_tr, X_val, y_val,
                              seen_cls, epochs, lr, batch_size, device,
                              es_patience=10, es_min_delta=1e-4):
    tr_mask  = np.isin(y_tr, seen_cls)
    X_t, y_t = X_tr[tr_mask], y_tr[tr_mask]
    val_mask = np.isin(y_val, seen_cls)
    X_v, y_v = X_val[val_mask], y_val[val_mask]

    n_total_cls  = clf.n_classes
    class_counts = np.array([max(1, (y_t == c).sum()) for c in range(n_total_cls)])
    class_weights = 1.0 / class_counts.astype(float)
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    ds      = MolDataset(X_t, y_t)
    dl      = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)
    opt     = torch.optim.Adam(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

    best_val_acc, best_weights, patience_count = -1.0, copy.deepcopy(clf.state_dict()), 0

    Xv_t = torch.tensor(X_v, dtype=torch.float32).to(device)
    yv_t = torch.tensor(y_v, dtype=torch.long).to(device)

    for epoch in range(epochs):
        clf.train()
        for Xb, yb in dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(clf(Xb), yb).backward()
            opt.step()

        clf.eval()
        with torch.no_grad():
            val_acc = (clf(Xv_t).argmax(1) == yv_t).float().mean().item()

        if val_acc > best_val_acc + es_min_delta:
            best_val_acc = val_acc; best_weights = copy.deepcopy(clf.state_dict()); patience_count = 0
        else:
            patience_count += 1
        if patience_count >= es_patience:
            break

    clf.load_state_dict(best_weights)
    clf.eval()
    return clf

print('train_classifier_on_task defined')

train_classifier_on_task defined


## Step 8 — ARC: Out-of-Task Detection (OTD)

### FIX-2 applied here: c_hat uses FULL softmax denominator

**Paper definition (page 5, Assumption 2):**
```
ĉ = max_{1 ≤ i ≤ s·(t−1)}  exp(z_i) / Σ_{j=1}^{s·t} exp(z_j)
```
The denominator is the **full** softmax sum over ALL classes (s·t), not just past classes.

**v9 bug**: `F.softmax(logits[:past_boundary])` renormalizes to sum=1 over only past logits,
making c_hat artificially large → w = c/c_hat always small → correction fires on everything.

**v10 fix**: Compute full softmax once, then take max over past-class indices.

In [18]:
def out_of_task_detection(logits, task_id, n_classes_per_task, epsilon, theta):
    """
    Paper-exact OTD — Assumptions 1 & 2 (ARC paper page 5).

    Assumption 1: pred ∈ PAST classes  AND  c >= ε  → retention
    Assumption 2: pred ∈ CURRENT classes  AND  w = c/c_hat < θ  → correction

    FIX-2 (v9→v10): c_hat is the max probability over PAST class indices
    in the FULL softmax distribution — NOT a renormalized softmax over only
    the past-logit slice. Using the truncated softmax inflates c_hat,
    making w always < θ and triggering correction on everything.

    Parameters
    ----------
    logits             : Tensor [N, n_total_classes]
    task_id            : int  0-based current task index
    n_classes_per_task : int  s
    epsilon            : float  Assumption 1 threshold
    theta              : float  Assumption 2 threshold

    Returns
    -------
    decisions : list[str]  'retention' | 'correction' | 'current'
    probs     : Tensor [N, n_total_classes]  full softmax probabilities
    pred_cls  : Tensor [N]  argmax predictions
    """
    s             = n_classes_per_task
    past_boundary = s * task_id    # classes 0..past_boundary-1 are past tasks

    # Full softmax — used for BOTH c (Assumption 1) and c_hat (Assumption 2)
    probs    = F.softmax(logits.float(), dim=-1)   # [N, C]
    pred_cls = probs.argmax(dim=-1)                # [N]
    conf_all = probs.max(dim=-1).values            # [N]  — this is c

    decisions = []
    for i in range(logits.shape[0]):
        pred_i = pred_cls[i].item()
        c      = probs[i, pred_i].item()

        # No past tasks at task 0 — everything is current
        if past_boundary == 0:
            decisions.append('current')
            continue

        if pred_i < past_boundary:
            # ── Assumption 1: predicted into a PAST class ─────────────────
            # c >= ε → model is confidently predicting a past class → retain
            if c >= epsilon:
                decisions.append('retention')
            else:
                decisions.append('current')

        else:
            # ── Assumption 2: predicted into a CURRENT-task class ─────────
            # c_hat = max probability over PAST class indices in FULL softmax
            # FIX-2: probs[i] is already the full softmax → slice, no re-normalize
            c_hat = probs[i, :past_boundary].max().item()   # ← FIXED (was F.softmax on slice)
            w     = c / (c_hat + 1e-8)
            # w < θ: model is more confident it is a past-task sample
            # than it is confident about the current-task prediction
            if w < theta:
                decisions.append('correction')
            else:
                decisions.append('current')

    return decisions, probs, pred_cls


print('OTD defined — FIX-2 applied (c_hat uses full softmax)')

OTD defined — FIX-2 applied (c_hat uses full softmax)


## Step 9 — Adaptive Retention

Equations 2 & 3 from the paper:
```
L_CE = -ŷ_c log p_c          (pseudo-label cross-entropy)
L_EM = -Σ p_i log p_i         (entropy minimisation)
L    = L_CE + L_EM
```
One gradient update per qualifying sample (online setting).

In [19]:
def adaptive_retention_step(clf, x_single, pseudo_label, lr, device):
    """
    One gradient update on the classifier head.
    L = L_CE(pseudo_label) + L_EM  (Eq. 2 & 3, ARC paper).

    NOTE: clf is the SHARED persistent_clf_arc object — updates accumulate
    across samples. Do NOT deepcopy clf before calling this.
    """
    clf.train()
    opt = torch.optim.SGD(clf.parameters(), lr=lr)
    opt.zero_grad()

    x       = x_single.detach().clone().to(device)
    logits  = clf(x)
    probs   = F.softmax(logits, dim=-1)
    label_t = torch.tensor([pseudo_label], dtype=torch.long).to(device)

    L_CE = F.cross_entropy(logits, label_t)
    L_EM = -(probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean()

    loss = L_CE + L_EM
    loss.backward()
    opt.step()
    clf.eval()


print('Adaptive Retention defined')

Adaptive Retention defined


## Step 10 — Adaptive Correction (TSS)

Definition 1 from the paper — Task-based Softmax Score:
```
S_i = max_{s(i-1) ≤ k < s·i}  exp(z_k / T^{t-i}) / Σ_{j=0}^{s·i-1} exp(z_j / T^{t-i})
```
Denominator = prefix sum over logits[0..s·i) with the SAME temperature as the numerator.

In [20]:
def compute_tss(logits_single, task_id, n_classes_per_task, temperature):
    """
    Task-based Softmax Score (TSS) — Definition 1 from ARC paper.

    For each past task i (1-based), compute:
      numerator   = max exp(z_k / T^{t-i})  for k in task i's class range
      denominator = sum exp(z_j / T^{t-i})  for j in [0, s*i)  (prefix)
      S_i         = numerator / denominator

    Returns the task and class with highest S_i.
    """
    s = n_classes_per_task
    t = task_id + 1   # 1-based current task
    z = logits_single.float()

    best_score, best_task, best_cls = -float('inf'), 0, 0

    for i in range(1, t ):          # i = 1..t (1-based)
        exponent  = t - i              # current task gets T^0 = 1; oldest gets T^{t-1}
        temp_i    = temperature ** exponent

        task_logits   = z[s * (i - 1) : s * i]    # logits for task i classes
        scaled_task   = task_logits / temp_i
        numerator     = scaled_task.exp().max()

        # Denominator: prefix [0, s*i) with same temperature  ← paper-exact
        prefix_logits = z[: s * i]
        denominator   = (prefix_logits / temp_i).exp().sum()

        score = (numerator / (denominator + 1e-12)).item()

        if score > best_score:
            best_score = score
            best_task  = i - 1                                      # back to 0-based
            best_cls   = s * (i - 1) + task_logits.argmax().item()

    return best_task, best_cls


print('Adaptive Correction (TSS) defined')

Adaptive Correction (TSS) defined


## Step 11 — Evaluation with ARC

### FIX-1 applied here: NO deepcopy of persistent_clf_arc inside evaluate_task

**v9 bug**: `clf_arc = copy.deepcopy(persistent_clf_arc)` was called on every evaluation call.
This meant all retention gradient updates were thrown away at the end of each `evaluate_task` call.
The persistent object never accumulated any updates — retention was completely no-op.

**v10 fix**: `persistent_clf_arc` is passed in and used directly (no deepcopy).
The caller (`run_cil_pipeline`) owns the object and snapshots it once per trained task.
Retention updates accumulate across all evaluation calls within that task's test stream.

In [21]:
def compute_auroc(y_t, probs_np, seen_cls, n_total_cls):
    valid_seen = [c for c in seen_cls if c < n_total_cls]
    if len(valid_seen) < 2:
        return float('nan')
    y_t_arr    = np.asarray(y_t)
    y_bin      = np.stack([(y_t_arr == c).astype(int) for c in valid_seen], axis=1)
    probs_seen = probs_np[:, valid_seen]
    probs_seen = probs_seen / (probs_seen.sum(axis=1, keepdims=True) + 1e-8)
    auroc_list = []
    for col_i in range(len(valid_seen)):
        col_true = y_bin[:, col_i]
        col_prob = probs_seen[:, col_i]
        if col_true.sum() == 0 or col_true.sum() == len(col_true):
            continue
        try:
            auroc_list.append(roc_auc_score(col_true, col_prob))
        except Exception:
            pass
    return float(np.mean(auroc_list)) if auroc_list else float('nan')


def evaluate_task(
    clf, X_test, y_test, seen_cls,
    current_task_id=None,    # trained task — sets OTD past/current boundary
    eval_task_id=None,       # task being evaluated (for logging only)
    n_cpt=None, epsilon=None, theta=None, temp=None,
    arc_lr=None, use_arc=False, device='cpu', persistent_clf_arc=None,
):
    """
    Evaluate classifier on test samples from seen_cls.

    FIX-1 (v9→v10): persistent_clf_arc is used DIRECTLY — no deepcopy.
      v9 deepcopied it on entry, discarding all retention updates after
      each call. Now retention updates accumulate across the entire
      test stream within a task boundary, exactly as Algorithm 1 specifies.

    FIX-2 (applied in out_of_task_detection): c_hat uses full softmax.
    """
    mask = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {'accuracy': 0.0, 'auroc': float('nan')}

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all = clf(Xt)
    n_total_cls = logits_all.shape[1]

    # ARC active only when there are past tasks (current_task_id > 0)
    if use_arc and current_task_id is not None and current_task_id > 0:

        # FIX-1: use persistent_clf_arc directly — NO deepcopy
        # Retention gradient steps accumulate across all samples and all
        # evaluation calls within this task boundary.
        clf_arc = persistent_clf_arc if persistent_clf_arc is not None \
                  else copy.deepcopy(clf)  # fallback only if caller didn't provide one

        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        preds, adapted_logits_list = [], []

        for i in range(len(Xt)):
            xi = Xt[i:i+1]
            with torch.no_grad():
                clf_arc.eval()
                li_otd = clf_arc(xi)

            # OTD with FIX-2 c_hat formula applied inside
            decisions, _, pred_i = out_of_task_detection(
                li_otd, current_task_id, n_cpt, epsilon, theta
            )
            decision     = decisions[0]
            pseudo_label = pred_i[0].item()
            otd_counts[decision] += 1

            if decision == 'retention':
                # One gradient update — updates clf_arc in-place
                adaptive_retention_step(clf_arc, xi, pseudo_label, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                pred_final = li_final.argmax(1).item()

            elif decision == 'correction':
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                _, pred_final = compute_tss(
                    li_final[0], current_task_id, n_cpt, temp
                )

            else:  # 'current' — accept as-is
                li_final   = li_otd
                pred_final = pseudo_label

            preds.append(pred_final)
            adapted_logits_list.append(li_final.detach())

        preds = np.array(preds)
        adapted_logits = torch.cat(adapted_logits_list, dim=0)
        with torch.no_grad():
            probs_np = F.softmax(adapted_logits, dim=-1).cpu().numpy()

        n = len(Xt)
        print(f'    OTD → retention:{otd_counts["retention"]}/{n}  '
              f'correction:{otd_counts["correction"]}/{n}  '
              f'current:{otd_counts["current"]}/{n}  '
              f'(ε={epsilon:.2f} ϑ={theta:.2f})')

    else:
        with torch.no_grad():
            preds    = logits_all.argmax(1).cpu().numpy()
            probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    accuracy = float((preds == y_t).mean())
    auroc    = compute_auroc(y_t, probs_np, seen_cls, n_total_cls)
    return {'accuracy': accuracy, 'auroc': auroc, 'preds': preds, 'labels': y_t}


print('evaluate_task defined (v10 — FIX-1 + FIX-2 applied)')

evaluate_task defined (v10 — FIX-1 + FIX-2 applied)


## Step 12 — Full CIL Pipeline

### FIX-4: persistent_clf_arc snapshot is used as-is (no deepcopy in evaluate_task)
### FIX-3: Diagnostic w computation uses corrected c_hat formula

In [22]:
def run_cil_pipeline(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, use_arc=False, name='Dataset'
):
    print(f'\n{"="*60}')
    print(f'  {name} CIL Pipeline  |  ARC={use_arc}')
    print(f'{"="*60}')
    print('  [Replay OFF] — memory-free for both baseline and ARC')

    clf                = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    R                  = defaultdict(dict)
    peak_acc           = {}
    rows               = []
    persistent_clf_arc = None   # FIX-4: single shared object; no deepcopy inside evaluate_task

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'\n-- Task {tid} | New classes: {[task["class_names"][c] for c in task["new_classes"]]} --')

        if tid > 0:
            clf.grow(len(task['new_classes']), device)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience  = cfg.get('es_patience', 10),
            es_min_delta = cfg.get('es_min_delta', 1e-4),
        )

        # ── Diagnostic block (Task 1 only) ─────────────────────────────────
        if tid == 1:
            import builtins
            builtins.clf_diag = copy.deepcopy(clf)
            print('  [diag] clf_diag saved to builtins')

            with torch.no_grad():
                _Xt = torch.tensor(X_te, dtype=torch.float32).to(device)
                _lg = clf(_Xt)
                _pr = F.softmax(_lg, dim=-1)            # full softmax
                _cf = _pr.max(dim=-1).values.cpu().numpy()   # c
                _pd = _lg.argmax(dim=-1).cpu().numpy()
            _pb = cfg['classes_per_task'] * 1   # past_boundary for task 1

            print(f"\n{'─'*50}")
            print(f'DIAGNOSTIC — confidence distribution (Task 1 clf)')
            print(f'  p10={np.percentile(_cf,10):.3f}  p25={np.percentile(_cf,25):.3f}')
            print(f'  p50={np.percentile(_cf,50):.3f}  p75={np.percentile(_cf,75):.3f}')
            print(f'  p90={np.percentile(_cf,90):.3f}  max={_cf.max():.3f}')
            for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
                print(f'  conf>{t:.2f}: {(_cf>t).sum():4d}/{len(_cf)} ({100*(_cf>t).mean():.1f}%)')

            _past = _pd < _pb
            print(f'\n  Past-class preds: {_past.sum()}/{len(_pd)}')
            if _past.sum() > 0:
                _cp = _cf[_past]
                print(f'  Past conf p50={np.percentile(_cp,50):.3f}  p75={np.percentile(_cp,75):.3f}')
                for t in [0.05, 0.10, 0.15, 0.20, 0.30]:
                    print(f'  past conf>{t:.2f}: {(_cp>t).sum():4d}/{_past.sum()} ({100*(_cp>t).mean():.1f}%)')

            _curr = _pd >= _pb
            print(f'\n  Current-class preds: {_curr.sum()}/{len(_pd)}')
            if _curr.sum() > 0:
                _widx = np.where(_curr)[0]
                _wv   = []
                _pr_np = _pr.cpu().numpy()   # full softmax probabilities
                for _i in _widx:
                    _c    = _cf[_i]                              # conf of predicted class
                    # FIX-3: c_hat = max over PAST indices in FULL softmax (not re-normalized slice)
                    _c_hat = _pr_np[_i, :_pb].max()             # ← FIXED
                    _wv.append(_c / (_c_hat + 1e-8))
                _wv = np.array(_wv)
                print(f'  w=c/c_hat: p25={np.percentile(_wv,25):.3f}  p50={np.percentile(_wv,50):.3f}  p75={np.percentile(_wv,75):.3f}')
                for t in [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]:
                    print(f'  w<{t:.1f}: {(_wv<t).sum():4d}/{len(_wv)} ({100*(_wv<t).mean():.1f}%)')
            print(f"{'─'*50}\n")

        # ── Snapshot clf for ARC AFTER training, BEFORE evaluation ────────
        # FIX-4: one snapshot per task; evaluate_task uses this object directly
        # (no deepcopy), so retention updates accumulate across all eval calls.
        if use_arc:
            persistent_clf_arc = copy.deepcopy(clf)

        # ── Evaluate all past + current tasks ─────────────────────────────
        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res = evaluate_task(
                clf, X_te, y_te, prev_seen,
                current_task_id    = tid,
                eval_task_id       = prev_tid,
                n_cpt              = cfg['classes_per_task'],
                epsilon            = cfg['arc_epsilon'],
                theta              = cfg['arc_theta'],
                temp               = cfg['arc_temp'],
                arc_lr             = cfg['arc_lr'],
                use_arc            = use_arc,
                device             = device,
                persistent_clf_arc = persistent_clf_arc,  # same object across all evals
            )
            R[tid][prev_tid] = res

            if prev_tid == tid:
                peak_acc[tid] = res['accuracy']

            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f'  Eval Task {prev_tid} → acc={res["accuracy"]:.4f}  auroc={auroc_str}')

    # ── Summary metrics ────────────────────────────────────────────────────
    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, auroc_list, forget_list = [], [], []

    for prev_tid in range(n_tasks):
        final_acc   = R[final_tid][prev_tid]['accuracy']
        final_auroc = R[final_tid][prev_tid]['auroc']
        acc_list.append(final_acc)
        auroc_list.append(final_auroc)
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - final_acc)

        rows.append({
            'task_id'    : prev_tid,
            'class_names': str(list(tasks[prev_tid]['class_names'].values())),
            'final_acc'  : round(final_acc, 4),
            'final_auroc': round(final_auroc, 4) if not np.isnan(final_auroc) else float('nan'),
            'peak_acc'   : round(peak_acc.get(prev_tid, final_acc), 4),
            'forgetting' : round(peak_acc.get(prev_tid, final_acc) - final_acc, 4),
        })

    avg_acc    = float(np.mean(acc_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*50}')
    print(f'  Average Accuracy (AB) : {avg_acc:.4f}')
    print(f'  Forgetting (F)        : {forgetting:.4f}')
    print(f'  Average AUROC         : {avg_auroc:.4f}')
    print(f'{"─"*50}')

    return pd.DataFrame(rows), avg_acc, forgetting, avg_auroc


print('CIL pipeline defined (v10 — all fixes applied)')

CIL pipeline defined (v10 — all fixes applied)


## Step 13 — Run: ChEMBL Baseline (No ARC)

In [23]:
chembl_res_no_arc, chembl_AB_no_arc, chembl_F_no_arc, chembl_AUROC_no_arc = run_cil_pipeline(
    tasks=chembl_tasks,
    X_tr=chembl_X_tr, y_tr=chembl_y_tr,
    X_val=chembl_X_vl, y_val=chembl_y_vl,
    X_te=chembl_X_te, y_te=chembl_y_te,
    feat_dim=FEAT_DIM, label_map=chembl_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='ChEMBL'
)
print('\nChEMBL Results (No ARC):')
print(chembl_res_no_arc.to_string(index=False))


  ChEMBL CIL Pipeline  |  ARC=False
  [Replay OFF] — memory-free for both baseline and ARC

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3663  auroc=0.5553

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  [diag] clf_diag saved to builtins

──────────────────────────────────────────────────
DIAGNOSTIC — confidence distribution (Task 1 clf)
  p10=0.233  p25=0.259
  p50=0.300  p75=0.355
  p90=0.426  max=0.827
  conf>0.05: 4455/4455 (100.0%)
  conf>0.10: 4455/4455 (100.0%)
  conf>0.15: 4455/4455 (100.0%)
  conf>0.20: 4425/4455 (99.3%)
  conf>0.30: 2215/4455 (49.7%)
  conf>0.50:  165/4455 (3.7%)

  Past-class preds: 1730/4455
  Past conf p50=0.300  p75=0.370
  past conf>0.05: 1730/1730 (100.0%)
  past conf>0.10: 1730/1730 (100.0%)
  past conf>0.15: 1730/1730 (100.0%)
  past conf>0.20: 1711/1730 (98.9%)
  past conf>0.30:  866/1730 (50.1%)

  Current-class preds: 2725/4455
  w=c/c_hat: p25=1.309  p50=1.737  p75=2.505
  w<0.5:    0/

## Step 14 — Run: ChEMBL + ARC

In [24]:
chembl_res_arc, chembl_AB_arc, chembl_F_arc, chembl_AUROC_arc = run_cil_pipeline(
    tasks=chembl_tasks,
    X_tr=chembl_X_tr, y_tr=chembl_y_tr,
    X_val=chembl_X_vl, y_val=chembl_y_vl,
    X_te=chembl_X_te, y_te=chembl_y_te,
    feat_dim=FEAT_DIM, label_map=chembl_label_map,
    cfg=CFG, device=DEVICE, use_arc=True, name='ChEMBL + ARC'
)
print('\nChEMBL Results (With ARC):')
print(chembl_res_arc.to_string(index=False))


  ChEMBL + ARC CIL Pipeline  |  ARC=True
  [Replay OFF] — memory-free for both baseline and ARC

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3697  auroc=0.5548

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  [diag] clf_diag saved to builtins

──────────────────────────────────────────────────
DIAGNOSTIC — confidence distribution (Task 1 clf)
  p10=0.214  p25=0.233
  p50=0.262  p75=0.301
  p90=0.346  max=0.566
  conf>0.05: 4455/4455 (100.0%)
  conf>0.10: 4455/4455 (100.0%)
  conf>0.15: 4455/4455 (100.0%)
  conf>0.20: 4315/4455 (96.9%)
  conf>0.30: 1137/4455 (25.5%)
  conf>0.50:   15/4455 (0.3%)

  Past-class preds: 1476/4455
  Past conf p50=0.253  p75=0.291
  past conf>0.05: 1476/1476 (100.0%)
  past conf>0.10: 1476/1476 (100.0%)
  past conf>0.15: 1476/1476 (100.0%)
  past conf>0.20: 1405/1476 (95.2%)
  past conf>0.30:  318/1476 (21.5%)

  Current-class preds: 2979/4455
  w=c/c_hat: p25=1.293  p50=1.648  p75=2.193
  w<0.5: 

## Step 15 — Run: BindingDB Baseline (No ARC)

In [25]:
bindingdb_res_no_arc, bindingdb_AB_no_arc, bindingdb_F_no_arc, bindingdb_AUROC_no_arc = run_cil_pipeline(
    tasks=bindingdb_tasks,
    X_tr=bindingdb_X_tr, y_tr=bindingdb_y_tr,
    X_val=bindingdb_X_vl, y_val=bindingdb_y_vl,
    X_te=bindingdb_X_te, y_te=bindingdb_y_te,
    feat_dim=FEAT_DIM, label_map=bindingdb_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='BindingDB'
)
print('\nBindingDB Results (No ARC):')
print(bindingdb_res_no_arc.to_string(index=False))


  BindingDB CIL Pipeline  |  ARC=False
  [Replay OFF] — memory-free for both baseline and ARC

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3817  auroc=0.5619

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  [diag] clf_diag saved to builtins

──────────────────────────────────────────────────
DIAGNOSTIC — confidence distribution (Task 1 clf)
  p10=0.221  p25=0.243
  p50=0.278  p75=0.326
  p90=0.379  max=0.807
  conf>0.05: 4423/4423 (100.0%)
  conf>0.10: 4423/4423 (100.0%)
  conf>0.15: 4423/4423 (100.0%)
  conf>0.20: 4355/4423 (98.5%)
  conf>0.30: 1600/4423 (36.2%)
  conf>0.50:   56/4423 (1.3%)

  Past-class preds: 1090/4423
  Past conf p50=0.267  p75=0.323
  past conf>0.05: 1090/1090 (100.0%)
  past conf>0.10: 1090/1090 (100.0%)
  past conf>0.15: 1090/1090 (100.0%)
  past conf>0.20: 1069/1090 (98.1%)
  past conf>0.30:  341/1090 (31.3%)

  Current-class preds: 3333/4423
  w=c/c_hat: p25=1.380  p50=1.851  p75=2.604
  w<0.5:   

## Step 16 — Run: BindingDB + ARC

In [1]:
bindingdb_res_arc, bindingdb_AB_arc, bindingdb_F_arc, bindingdb_AUROC_arc = run_cil_pipeline(
    tasks=bindingdb_tasks,
    X_tr=bindingdb_X_tr, y_tr=bindingdb_y_tr,
    X_val=bindingdb_X_vl, y_val=bindingdb_y_vl,
    X_te=bindingdb_X_te, y_te=bindingdb_y_te,
    feat_dim=FEAT_DIM, label_map=bindingdb_label_map,
    cfg=CFG, device=DEVICE, use_arc=True, name='BindingDB + ARC'
)
print('\nBindingDB Results (With ARC):')
print(bindingdb_res_arc.to_string(index=False))

NameError: name 'run_cil_pipeline' is not defined

## Step 17 — Comparison Summary

In [27]:
print('=' * 60)
print('  FINAL COMPARISON SUMMARY')
print('=' * 60)
print(f"{'Dataset':<20} {'Method':<12} {'AB':>8} {'Forgetting':>12} {'AUROC':>8}")
print('-' * 60)
print(f"{'ChEMBL':<20} {'Baseline':<12} {chembl_AB_no_arc:>8.4f} {chembl_F_no_arc:>12.4f} {chembl_AUROC_no_arc:>8.4f}")
print(f"{'ChEMBL':<20} {'+ ARC':<12} {chembl_AB_arc:>8.4f} {chembl_F_arc:>12.4f} {chembl_AUROC_arc:>8.4f}")
print(f"{'ChEMBL':<20} {'Δ (ARC-Base)':<12} {chembl_AB_arc-chembl_AB_no_arc:>+8.4f} {chembl_F_arc-chembl_F_no_arc:>+12.4f} {chembl_AUROC_arc-chembl_AUROC_no_arc:>+8.4f}")
print('-' * 60)
print(f"{'BindingDB':<20} {'Baseline':<12} {bindingdb_AB_no_arc:>8.4f} {bindingdb_F_no_arc:>12.4f} {bindingdb_AUROC_no_arc:>8.4f}")
print(f"{'BindingDB':<20} {'+ ARC':<12} {bindingdb_AB_arc:>8.4f} {bindingdb_F_arc:>12.4f} {bindingdb_AUROC_arc:>8.4f}")
print(f"{'BindingDB':<20} {'Δ (ARC-Base)':<12} {bindingdb_AB_arc-bindingdb_AB_no_arc:>+8.4f} {bindingdb_F_arc-bindingdb_F_no_arc:>+12.4f} {bindingdb_AUROC_arc-bindingdb_AUROC_no_arc:>+8.4f}")
print('=' * 60)
print('  Expected: ARC AB > Baseline, ARC Forgetting < Baseline, ARC AUROC > Baseline')

  FINAL COMPARISON SUMMARY
Dataset              Method             AB   Forgetting    AUROC
------------------------------------------------------------
ChEMBL               Baseline       0.1306       0.0915   0.5834
ChEMBL               + ARC          0.1109       0.0848   0.5344
ChEMBL               Δ (ARC-Base)  -0.0196      -0.0067  -0.0489
------------------------------------------------------------
BindingDB            Baseline       0.1343       0.0844   0.5848
BindingDB            + ARC          0.1195       0.0512   0.4908
BindingDB            Δ (ARC-Base)  -0.0148      -0.0332  -0.0941
  Expected: ARC AB > Baseline, ARC Forgetting < Baseline, ARC AUROC > Baseline
